<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1C: use csv file to train ML model and test accuracy

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [43]:
from datetime import datetime
import geopandas as gpd
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


<font color='red'>1.1. Input your initials and site name below. Consider how this should align with the files you are trying to open: 


In [44]:
initials = "nm"
site = "usp"
filename = f"{site}-tdata.csv"
version = f"{initials}-lulc-{site}"

### Postcard csv

The objective of this notebook was to train the machine learning model that will allow us to classify an area with land cover classes defined through the training data.

Step 1.2. Input the training data to sample geomad data from the postcard

In [45]:
joined_df = gpd.read_file(f"training-data/{filename}")
# joined_df

In [48]:
joined_df

,LULC_code,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,...,swir22_swir16,mci,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x
0,5.0,0.25070000000000003,0.159,0.12760000000000002,0.15710000000000002,0.07737054,8.668303e-08,4.5086913e-06,0.3251,0.2923,...,0.6977578475336323,1.483431952662722,0.030487804878048808,-0.05847582858349172,0.05847582858349172,-0.09193245778611644,-0.15129999999999988,-0.0016950699999999722,-2041715.0,3167305.0
1,3.0,0.43270000000000003,0.056,0.0521,0.0911,0.09050532,8.7711214e-08,4.9841133e-06,0.35100000000000003,0.3126,...,0.6525842696629213,2.896251673360107,0.4547224926971763,-0.32081807081807084,0.32081807081807084,-0.41341107871720123,-0.66665,-0.08151828000000001,-2041705.0,3167325.0
2,5.0,0.34800000000000003,0.079,0.0655,0.0944,0.083218575,1.4163851e-07,4.6851237e-06,0.37270000000000003,0.3672,...,0.6796815769522365,1.9594594594594597,0.3842556508183944,-0.1376266753841124,0.1376266753841124,-0.23102113724322715,-0.5713750000000002,-0.08861235000000003,-2041735.0,3167335.0
3,3.0,0.3855,0.0645,0.053000000000000005,0.0848,0.07680272,1.6126334e-07,4.7224207e-06,0.378,0.3658,...,0.6444545869465997,2.3321234119782215,0.4386422976501305,-0.27522328812437974,0.27522328812437974,-0.36137845389630546,-0.6066,-0.06734341,-2041755.0,3167355.0
4,5.0,0.3617,0.08700000000000001,0.0674,0.0984,0.0857967,2.1393895e-07,5.1306342e-06,0.41450000000000004,0.37020000000000003,...,0.5893574297188755,2.559801840056617,0.2378449408672799,-0.28971296131217683,0.28971296131217683,-0.3546961325966851,-0.49860000000000004,-0.046525189999999994,-2041765.0,3167365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,3.0,0.46390000000000003,0.0806,0.0723,0.12150000000000001,0.13946256,1.296997e-07,5.7257157e-06,0.5096,0.45270000000000005,...,0.5259992925362575,2.7018054746651137,0.36107808164883076,-0.24270024109295474,0.24270024109295474,-0.4009144350097975,-0.7066750000000002,-0.08580073999999996,-2041815.0,3167465.0
357,3.0,0.47340000000000004,0.079,0.0651,0.11470000000000001,0.08140515,1.3719499e-07,3.570803e-06,0.5169,0.40850000000000003,...,0.5171065493646139,2.5826513911620292,0.39763629431948155,-0.2133794694348328,0.2133794694348328,-0.38752898737438807,-0.7789250000000001,-0.09922040999999998,-2041805.0,3167515.0
358,6.0,0.16720000000000002,0.0819,0.07250000000000001,0.09290000000000001,0.09434129,7.9820154e-07,1.150706e-05,0.1338,0.26230000000000003,...,0.6457680250783698,1.8557158712541622,0.04767441860465116,-0.2719665271966527,0.2719665271966527,-0.2503912363067293,-0.07415,0.022940740000000005,-2041825.0,3167565.0
359,6.0,0.10650000000000001,0.0792,0.07200000000000001,0.0887,0.08847977,5.522847e-07,8.400731e-06,0.1875,0.40080000000000005,...,0.625,0.9517426273458446,0.17111459968602824,0.08544439673679691,-0.08544439673679691,-0.06029106029106033,-0.03585000000000005,0.013246930000000018,-2041835.0,3167545.0


In [49]:
joined_df = joined_df.astype("float32")
joined_df


,LULC_code,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,...,swir22_swir16,mci,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x
0,5.0,0.2507,0.1590,0.1276,0.1571,0.077371,8.668303e-08,0.000005,0.3251,0.2923,...,0.697758,1.483432,0.030488,-0.058476,0.058476,-0.091932,-0.151300,-0.001695,-2041715.0,3167305.0
1,3.0,0.4327,0.0560,0.0521,0.0911,0.090505,8.771121e-08,0.000005,0.3510,0.3126,...,0.652584,2.896252,0.454722,-0.320818,0.320818,-0.413411,-0.666650,-0.081518,-2041705.0,3167325.0
2,5.0,0.3480,0.0790,0.0655,0.0944,0.083219,1.416385e-07,0.000005,0.3727,0.3672,...,0.679682,1.959459,0.384256,-0.137627,0.137627,-0.231021,-0.571375,-0.088612,-2041735.0,3167335.0
3,3.0,0.3855,0.0645,0.0530,0.0848,0.076803,1.612633e-07,0.000005,0.3780,0.3658,...,0.644455,2.332124,0.438642,-0.275223,0.275223,-0.361378,-0.606600,-0.067343,-2041755.0,3167355.0
4,5.0,0.3617,0.0870,0.0674,0.0984,0.085797,2.139389e-07,0.000005,0.4145,0.3702,...,0.589357,2.559802,0.237845,-0.289713,0.289713,-0.354696,-0.498600,-0.046525,-2041765.0,3167365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,3.0,0.4639,0.0806,0.0723,0.1215,0.139463,1.296997e-07,0.000006,0.5096,0.4527,...,0.525999,2.701806,0.361078,-0.242700,0.242700,-0.400914,-0.706675,-0.085801,-2041815.0,3167465.0
357,3.0,0.4734,0.0790,0.0651,0.1147,0.081405,1.371950e-07,0.000004,0.5169,0.4085,...,0.517107,2.582651,0.397636,-0.213379,0.213379,-0.387529,-0.778925,-0.099220,-2041805.0,3167515.0
358,6.0,0.1672,0.0819,0.0725,0.0929,0.094341,7.982015e-07,0.000012,0.1338,0.2623,...,0.645768,1.855716,0.047674,-0.271967,0.271967,-0.250391,-0.074150,0.022941,-2041825.0,3167565.0
359,6.0,0.1065,0.0792,0.0720,0.0887,0.088480,5.522847e-07,0.000008,0.1875,0.4008,...,0.625000,0.951743,0.171115,0.085444,-0.085444,-0.060291,-0.035850,0.013247,-2041835.0,3167545.0


<font color='red'>1.2. Delete the columns that you will not need including y and x by inputting them below: 
>joined_df=joined_df.drop(columns=["`...`", "`...`"])

In [50]:
joined_df=joined_df.drop(columns=["y", "x"])

<font color='red'>1.3. Print the length or number of columns using variable.columns of the new dataset: 
>print(len(`variable`.`...`))

In [51]:
len(joined_df)

361

<font color='red'>1.4. Print the variable.columns of the new dataset: 
>`variable`.columns

In [52]:
joined_df.columns

Index(['LULC_code', 'nir', 'red', 'blue', 'green', 'emad', 'smad', 'bcmad',
       'nir08', 'nir09', 'swir16', 'swir22', 'coastal', 'rededge1', 'rededge2',
       'rededge3', 'mndwi', 'ndti', 'cai', 'ndvi', 'evi', 'savi', 'ndwi',
       'b_g', 'b_r', 'swir22_swir16', 'mci', 'ndci', 'nbi', 'ndmi', 'bsi',
       'awei', 'tc_wetness'],
      dtype='object')

<font color='red'>1.5. Run the machine learning on your joined_df variable data: 
>training_data, test_data = train_test_split(`variable`, test_size=0.2, random_state=1337)


In [53]:
training_data, test_data = train_test_split(joined_df, test_size=0.2, random_state=1337)

<font color='red'>1.5. Run the machine learning on your `training_data` variable by replacing `variable` below: 
>training_data, test_data = train_test_split(`variable`, test_size=0.2, random_state=1337)


In [54]:
# The classes are the first column
classes = np.array(joined_df)[:, 0]

# The observation data is everything after the second column
observations = np.array(joined_df)[:, 1:]

# Create a model...
classifier = RandomForestClassifier(max_depth=4)

# ...and fit it to the data
model = classifier.fit(observations, classes)

<font color='red'>1.6. Export the model file

In [55]:
# Dynamically create the filename with f-string
file_path = f"models/{version}-test.model"

# Save the model
joblib.dump(model, file_path)

['models/nm-lulc-usp-test.model']

<font color='red'>1.6. Complete a confusion matrix and display it. Discuss what this might mean with the person next to you.

In [56]:
import pandas as pd
test_actual = np.array(test_data)[:, 0]

test_predicted = model.predict(np.array(test_data)[:,1:])


unique_labels = sorted(np.unique(np.concatenate([np.asarray(test_actual), np.asarray(test_predicted)])))


pd.crosstab(test_actual, test_predicted, margins=True)

col_0,2.0,3.0,5.0,6.0,All
row_0,,,,,
2.0,9,1,5,0,15
3.0,1,17,2,0,20
5.0,4,1,26,0,31
6.0,0,1,4,2,7
All,14,20,37,2,73


<font color='red'>1.6. Complete an accuracy score and display it. Discuss what this might mean with the person next to you.

In [57]:
from sklearn.metrics import accuracy_score

accuracy_score(test_actual, test_predicted)

0.7397260273972602